<a href="https://colab.research.google.com/github/shivamkumar359/03-rd-SEM--ASSIGNMENTS/blob/main/IDB/W10/week_10_practical_session_Q.1%20.ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from dataclasses import dataclass
from pathlib import Path
import zipfile
import subprocess
import sys

In [2]:
# --------------------------------------------------
# PART 1: DATA STRUCTURES
# --------------------------------------------------

@dataclass
class Transaction:
    sequence: int
    transaction_id: str
    timestamp: int
    operation: str
    data_item: str


@dataclass
class DataItem:
    name: str
    read_timestamp: int = 0
    write_timestamp: int = 0



In [3]:

# --------------------------------------------------
# PART 2: SAMPLE DATASET
# --------------------------------------------------

transactions = [
    Transaction(1, "T1", 10, "READ", "Q"),
    Transaction(2, "T2", 20, "WRITE", "Q"),
    Transaction(3, "T3", 15, "READ", "Q"),
    Transaction(4, "T4", 25, "WRITE", "Q"),
    Transaction(5, "T5", 18, "WRITE", "Q")
]

In [4]:
# --------------------------------------------------
# PART 3: READ OPERATION
# --------------------------------------------------

def read_operation(transaction, item):

    ts = transaction.timestamp

    if ts < item.write_timestamp:

        return "REJECT - ROLLBACK"

    else:

        item.read_timestamp = max(
            item.read_timestamp,
            ts
        )

        return "GRANT"

In [5]:
# --------------------------------------------------
# PART 4: WRITE OPERATION
# --------------------------------------------------

def write_operation(transaction, item):

    ts = transaction.timestamp

    if ts < item.read_timestamp:

        return "REJECT - ROLLBACK"

    elif ts < item.write_timestamp:

        return "REJECT - ROLLBACK"

    else:

        item.write_timestamp = ts

        return "GRANT"

In [6]:
# --------------------------------------------------
# PART 5: PROCESS TRANSACTIONS
# --------------------------------------------------

def process_transaction(transaction, item):

    if transaction.data_item != item.name:
        return "ERROR - UNKNOWN DATA ITEM"

    if transaction.operation == "READ":

        return read_operation(transaction, item)

    elif transaction.operation == "WRITE":

        return write_operation(transaction, item)

    else:

        return "ERROR - UNKNOWN OPERATION"

In [7]:
# --------------------------------------------------
# PART 6: EXECUTION LOG
# --------------------------------------------------

def execute_protocol():

    item = DataItem("Q")

    results = []

    results.append(
        "TIMESTAMP-BASED CONCURRENCY CONTROL"
    )

    results.append(
        "=" * 100
    )

    results.append(
        f"Initial R-TS(Q): {item.read_timestamp}"
    )

    results.append(
        f"Initial W-TS(Q): {item.write_timestamp}"
    )

    results.append("")

    header = (
        f"{'Seq':<6}"
        f"{'Transaction':<15}"
        f"{'Timestamp':<12}"
        f"{'Operation':<12}"
        f"{'Item':<8}"
        f"{'Decision':<24}"
        f"{'R-TS(Q)':<12}"
        f"{'W-TS(Q)':<12}"
    )

    results.append(header)
    results.append("-" * 100)

    for transaction in transactions:

        decision = process_transaction(
            transaction,
            item
        )

        row = (
            f"{transaction.sequence:<6}"
            f"{transaction.transaction_id:<15}"
            f"{transaction.timestamp:<12}"
            f"{transaction.operation:<12}"
            f"{transaction.data_item:<8}"
            f"{decision:<24}"
            f"{item.read_timestamp:<12}"
            f"{item.write_timestamp:<12}"
        )

        results.append(row)

    results.append("-" * 100)

    results.append(
        f"Final R-TS(Q): {item.read_timestamp}"
    )

    results.append(
        f"Final W-TS(Q): {item.write_timestamp}"
    )

    return "\n".join(results)

In [9]:
# --------------------------------------------------
# PART 7: CREATE TERMINAL SCREENSHOT
# --------------------------------------------------

def create_screenshot(output_text, output_dir):

    try:

        from PIL import Image, ImageDraw, ImageFont

    except ImportError:

        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "pillow"
        ])

        from PIL import Image, ImageDraw, ImageFont

    font = ImageFont.load_default()

    lines = [
        "Google Colab - Timestamp Protocol",
        "$ python timestamp_protocol.py"
    ] + output_text.splitlines()

    width = 1250
    line_height = 25
    height = len(lines) * line_height + 50

    image = Image.new(
        "RGB",
        (width, height),
        "#101418"
    )

    draw = ImageDraw.Draw(image)

    draw.rectangle(
        (0, 0, width, 35),
        fill="#252a31"
    )

    y = 45

    for line in lines:

        if "GRANT" in line:
            color = "#98c379"

        elif "REJECT" in line:
            color = "#e06c75"

        else:
            color = "#ffffff"

        draw.text(
            (20, y),
            line,
            fill=color,
            font=font
        )

        y += line_height

    image.save(
        output_dir / "timestamp_protocol_output.png"
    )

In [10]:
# --------------------------------------------------
# PART 8: CREATE DEADLOCK ANALYSIS PDF
# --------------------------------------------------

def create_pdf(output_dir):

    try:

        from reportlab.lib.pagesizes import A4
        from reportlab.platypus import (
            SimpleDocTemplate,
            Paragraph,
            Spacer
        )
        from reportlab.lib.styles import getSampleStyleSheet
        from reportlab.lib.units import mm

    except ImportError:

        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "reportlab"
        ])

        from reportlab.lib.pagesizes import A4
        from reportlab.platypus import (
            SimpleDocTemplate,
            Paragraph,
            Spacer
        )
        from reportlab.lib.styles import getSampleStyleSheet
        from reportlab.lib.units import mm

    report = """
    The Timestamp Ordering Protocol assigns a unique timestamp
    to every transaction and ensures that conflicting operations
    follow this timestamp order.

    A read operation is rejected if its transaction timestamp
    is smaller than the Write-Timestamp (W-TS) of the data item.

    A write operation is rejected if its timestamp is smaller
    than either the Read-Timestamp (R-TS) or Write-Timestamp
    (W-TS) of the data item.

    Rejected transactions are rolled back to maintain consistency.
    Successful operations update the corresponding timestamps.

    Therefore, every edge in the precedence graph points from
    an older transaction to a newer transaction.

    Since timestamps must strictly increase along every edge,
    a directed cycle is impossible. A cycle would require the
    timestamps to increase and eventually return to their
    original value, which cannot happen.

    Thus, the precedence graph remains acyclic, preventing
    deadlocks caused by cyclic transaction dependencies.
    """

    styles = getSampleStyleSheet()

    document = SimpleDocTemplate(
        str(output_dir / "deadlock_analysis.pdf"),
        pagesize=A4,
        rightMargin=22 * mm,
        leftMargin=22 * mm
    )

    content = [
        Paragraph(
            "Deadlock Analysis: Timestamp Ordering Protocol",
            styles["Title"]
        ),

        Spacer(1, 15),

        Paragraph(
            report,
            styles["BodyText"]
        )
    ]

    document.build(content)




In [11]:
# --------------------------------------------------
# PART 9: MAIN PROGRAM
# --------------------------------------------------

def main():

    output_dir = Path.cwd()

    # Execute protocol
    output = execute_protocol()

    # Display terminal output
    print(output)

    # Generate execution log
    log_file = output_dir / "protocol_results.txt"

    log_file.write_text(
        output,
        encoding="utf-8"
    )

    # Generate screenshot
    create_screenshot(
        output,
        output_dir
    )

    # Generate PDF report
    create_pdf(
        output_dir
    )

    # Generate ZIP archive
    zip_file = output_dir / "TASK_1_TIMESTAMP_PROTOCOL.zip"

    required_files = [
        "timestamp_protocol.py",
        "protocol_results.txt",
        "timestamp_protocol_output.png",
        "deadlock_analysis.pdf"
    ]

    with zipfile.ZipFile(
        zip_file,
        "w",
        zipfile.ZIP_DEFLATED
    ) as archive:

        for filename in required_files:

            file_path = output_dir / filename

            if file_path.exists():

                archive.write(
                    file_path,
                    arcname=filename
                )

    print("\nALL DELIVERABLES GENERATED SUCCESSFULLY!")

    for filename in required_files:
        print("Created:", filename)

    print("Created:", zip_file.name)


if __name__ == "__main__":

    main()

TIMESTAMP-BASED CONCURRENCY CONTROL
Initial R-TS(Q): 0
Initial W-TS(Q): 0

Seq   Transaction    Timestamp   Operation   Item    Decision                R-TS(Q)     W-TS(Q)     
----------------------------------------------------------------------------------------------------
1     T1             10          READ        Q       GRANT                   10          0           
2     T2             20          WRITE       Q       GRANT                   10          20          
3     T3             15          READ        Q       REJECT - ROLLBACK       10          20          
4     T4             25          WRITE       Q       GRANT                   10          25          
5     T5             18          WRITE       Q       REJECT - ROLLBACK       10          25          
----------------------------------------------------------------------------------------------------
Final R-TS(Q): 10
Final W-TS(Q): 25

ALL DELIVERABLES GENERATED SUCCESSFULLY!
Created: timestamp_protocol.py
Cr